# Anomaly detection — end-to-end, against the real `landed.duckdb` history

This notebook exercises the **current, shipped** implementation —
`app/service/pipeline/anomaly.py`'s `detect_anomalies` (statistical
z-score signal) and `app/service/pipeline/ml_anomaly.py` (the trained
`IsolationForest` ML signal) — imported directly from the app, not
re-derived here. Nothing below is a prototype; it's the real production
code, run end-to-end against the real accumulated history sitting in
`services/ingestion/data/staging/landed.duckdb`.

**Current signal design** (rule-based checks were retired 2026-08-12 —
see `anomaly.py`'s own docstring for why: the missing-value check alone
used to account for 121K/150K+ real `meta.anomalies` rows, almost all
structurally expected rather than anomalous):

1. **Statistical**: a per-batch z-score against that column's own
   mean/std within the batch being scanned (`_Z_SCORE_THRESHOLD = 4.0`).
2. **ML**: a per-source `IsolationForest`, trained against that source's
   full accumulated staging history (`ecolens-ingestion
   train-anomaly-model <source>`), scored per row
   (`ANOMALY_SCORE_THRESHOLD = 0.7` of its own calibrated
   threshold→floor range). `None` (no contribution) until a model has
   actually been trained for a source.

Both signals feed one **high-confidence-only gate**
(`_MIN_ANOMALY_SCORE_TO_FLAG = 0.98`): clearing a signal's own trigger
is necessary but not sufficient — the winning signal's own combined
score must also clear 0.98 for a row to actually get flagged.

Trained model artifacts already exist on disk for every source that has
one (`data/staging/models/anomaly/*.joblib`) — this notebook scores
against them as-is; it does not retrain (`ecolens-ingestion
train-anomaly-model` is the real retraining entrypoint, and would
overwrite the local file + re-upload to R2, not something to do
casually from a notebook run).

In [7]:
!uv pip install openelectricity
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Same "walk up to find services/ingestion/.env" pattern as
# notebooks/EDA.ipynb's setup cell.
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

# **Must happen before the first `get_settings()` call anywhere** (it's
# `@lru_cache`d, so whichever value is in the environment at that first
# call wins for the rest of the process). `Settings.duckdb_staging_dir`
# defaults to the relative `"./data/staging"` -- fine for `Settings`
# itself (its `env_file` is pinned to an absolute path via `_PACKAGE_
# ROOT`, so `.env` loading doesn't care about cwd), but every *downstream*
# consumer (`duckdb_staging._staging_path`, `ml_anomaly._model_dir`)
# resolves that relative string against the process's cwd at the point
# of use -- and Jupyter's default execution cwd is this notebook's own
# directory (`services/ingestion/notebooks/`), not `services/ingestion/`.
# Left unfixed, every real-data cell below silently "succeeds" against
# an empty/nonexistent `notebooks/data/staging/` instead of the real
# staging file, and every ML model load silently returns `None` (cached
# as `None` for the rest of the process) -- no error, just quietly wrong
# (empty) results throughout. Overriding the env var here, before
# anything else imports/calls `get_settings()`, makes the resolved path
# absolute and correct everywhere downstream, unconditionally.
os.environ["DUCKDB_STAGING_DIR"] = str(INGESTION_DIR / "data" / "staging")

from app.core.config import get_settings  # noqa: E402
from app.service.pipeline import anomaly, ml_anomaly  # noqa: E402
from app.service.pipeline.duckdb_staging import read_table_history  # noqa: E402
from app.service.pipeline.tasks.registry import SOURCES  # noqa: E402

settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
assert db_path.is_absolute(), f"DUCKDB_STAGING_DIR override didn't take -- got {db_path}"
db_path = db_path / "landed.duckdb"
assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

print("loaded .env from:", INGESTION_DIR / ".env")
print("staging file:", db_path)
print("registry sources:", list(SOURCES.keys()))
print("anomaly-scanned columns per source:", anomaly._NUMERIC_COLUMNS)

Using Python 3.13.9 environment at: /Users/macbook/Project/research/EcoLens/services/forecast-api/.venv
Audited 1 package in 16ms
loaded .env from: /Users/macbook/Project/research/EcoLens/services/ingestion/.env
staging file: /Users/macbook/Project/research/EcoLens/services/ingestion/data/staging/landed.duckdb
registry sources: ['oe', 'aemo-nem', 'aemo-wem', 'bom', 'holidays']
anomaly-scanned columns per source: {'openelectricity': ('total_generation_mw',), 'aemo_nem': ('demand_mw', 'price_mwh'), 'aemo_wem': ('demand_mw', 'price_mwh'), 'bom': ('temp_c', 'humidity_pct', 'wind_speed_kmh'), 'aemo_holidays': ()}


## Trained ML models on disk

`ml_anomaly.load_local(source)` reads straight from
`data/staging/models/anomaly/{source}.joblib`, bypassing the in-process
score cache — the same artifact `ecolens-ingestion train-anomaly-model`
produces and `ml_anomaly.score` (via `detect_anomalies`) loads at
serving time. `rows_trained`/`decision_threshold`/`decision_floor` are
each model's own real calibration (see `ml_anomaly.py`'s module
docstring for why these are per-model, not a universal constant).

In [8]:
# The 4 sources with numeric columns worth scanning (aemo_holidays has
# none -- ml_anomaly.train() returns None for it, see anomaly._NUMERIC_COLUMNS).
_ML_SOURCES = [s for s, cols in anomaly._NUMERIC_COLUMNS.items() if cols]

model_rows = []
for source in _ML_SOURCES:
    model = ml_anomaly.load_local(source)
    if model is None:
        model_rows.append({"source": source, "trained": False})
        continue
    model_rows.append(
        {
            "source": source,
            "trained": True,
            "columns": model.columns,
            "rows_trained": model.rows_trained,
            "decision_threshold": round(model.decision_threshold, 4),
            "decision_floor": round(model.decision_floor, 4),
        }
    )

models_df = pd.DataFrame(model_rows)
models_df

,source,trained,columns,rows_trained,decision_threshold,decision_floor
0,openelectricity,True,"(total_generation_mw,)",21983,0.0,-0.0451
1,aemo_nem,True,"(demand_mw, price_mwh)",61080,0.0,-0.0512
2,aemo_wem,True,"(demand_mw, price_mwh)",8088,0.0,-0.0755
3,bom,True,"(temp_c, humidity_pct, wind_speed_kmh)",141876,-0.0,-0.0523


## Run `detect_anomalies` against each source's full real history

Each source's entire accumulated `landed.duckdb` history (`read_table_
history` — the exact same read `ml_anomaly.train` itself uses) is
scanned in one batch through the real `anomaly.detect_anomalies`. This
means the **statistical** signal's mean/std is computed over that whole
multi-year history rather than one live ~5-min fetch batch (production
scans one small live batch at a time) — a deliberately different,
harder framing than production for this end-to-end run, while the
**ML** signal behaves identically either way (it's already scored
per-row against a model trained on the full history, batch size
doesn't change its calibration).

In [9]:
results = {}  # source -> (history_df, flags_df)


def run_end_to_end(source: str, table: str) -> pd.DataFrame:
    """Real `read_table_history` + real `anomaly.detect_anomalies`,
    with a compact per-source summary printed (row count, flag count/
    rate, and a breakdown of which signal(s) fired)."""
    history = read_table_history(table)
    flags = anomaly.detect_anomalies(history, source)
    results[source] = (history, flags)

    rate = len(flags) / max(len(history), 1)
    print(f"{source} ({table}): {len(history):,} rows scanned, {len(flags)} flagged ({rate:.4%})")
    if not flags.empty:
        stat_only = flags["anomaly_statistical_score"].notna() & flags["anomaly_ml_score"].isna()
        ml_only = flags["anomaly_ml_score"].notna() & flags["anomaly_statistical_score"].isna()
        both = flags["anomaly_statistical_score"].notna() & flags["anomaly_ml_score"].notna()
        print(f"  statistical-only: {stat_only.sum()}, ml-only: {ml_only.sum()}, both: {both.sum()}")
        print(flags[["anomaly_reason", "anomaly_score"]].head(10).to_string())
    return flags


# bom -- temp_c/humidity_pct/wind_speed_kmh
bom_flags = run_end_to_end("bom", "bom_observations")

bom (bom_observations): 237,570 rows scanned, 204 flagged (0.0859%)
  statistical-only: 0, ml-only: 176, both: 28
                                anomaly_reason  anomaly_score
6920   ml_outlier:isolation_forest(score=1.00)       1.000000
7780   ml_outlier:isolation_forest(score=1.00)       1.000000
7784   ml_outlier:isolation_forest(score=1.00)       1.000000
7785   ml_outlier:isolation_forest(score=1.00)       1.000000
9627   ml_outlier:isolation_forest(score=1.00)       1.000000
9628   ml_outlier:isolation_forest(score=1.00)       1.000000
10926  ml_outlier:isolation_forest(score=0.98)       0.982268
10972  ml_outlier:isolation_forest(score=1.00)       1.000000
10973  ml_outlier:isolation_forest(score=1.00)       1.000000
11441  ml_outlier:isolation_forest(score=1.00)       1.000000


In [10]:
# aemo_nem -- demand_mw/price_mwh
nem_flags = run_end_to_end("aemo_nem", "aemo_nem_dispatch")

aemo_nem (aemo_nem_dispatch): 1,358,005 rows scanned, 6937 flagged (0.5108%)
  statistical-only: 0, ml-only: 5378, both: 1559
                                                                      anomaly_reason  anomaly_score
1365  statistical_outlier:price_mwh(z=23.5); ml_outlier:isolation_forest(score=1.00)            1.0
1436  statistical_outlier:price_mwh(z=35.8); ml_outlier:isolation_forest(score=1.00)            1.0
1438                                         ml_outlier:isolation_forest(score=1.00)            1.0
1443                                         ml_outlier:isolation_forest(score=1.00)            1.0
2392                                         ml_outlier:isolation_forest(score=1.00)            1.0
3144                                         ml_outlier:isolation_forest(score=1.00)            1.0
3145                                         ml_outlier:isolation_forest(score=1.00)            1.0
3147                                         ml_outlier:isolation_forest(s

In [11]:
# aemo_wem -- demand_mw/price_mwh
wem_flags = run_end_to_end("aemo_wem", "aemo_wem_dispatch")

aemo_wem (aemo_wem_dispatch): 211,104 rows scanned, 110 flagged (0.0521%)
  statistical-only: 73, ml-only: 5, both: 32
                                                                      anomaly_reason  anomaly_score
729                                            statistical_outlier:price_mwh(z=11.6)            1.0
731                                            statistical_outlier:price_mwh(z=11.7)            1.0
762                                            statistical_outlier:price_mwh(z=13.1)            1.0
1084                                           statistical_outlier:price_mwh(z=13.2)            1.0
1137                                           statistical_outlier:price_mwh(z=13.2)            1.0
1522                                            statistical_outlier:price_mwh(z=9.8)            1.0
9219                                           statistical_outlier:price_mwh(z=13.2)            1.0
9253                                           statistical_outlier:price_mwh(z=13

In [12]:
# openelectricity -- total_generation_mw only. demand_mw/price_mwh are
# deliberately excluded (see anomaly.py's own module docstring: a past
# real incident where those two columns were structurally always None
# and flooded meta.anomalies with ~100% of every batch).
oe_flags = run_end_to_end("openelectricity", "openelectricity_mix")

openelectricity (openelectricity_mix): 1,270,957 rows scanned, 7813 flagged (0.6147%)
  statistical-only: 0, ml-only: 7813, both: 0
                                anomaly_reason  anomaly_score
2557   ml_outlier:isolation_forest(score=1.00)            1.0
2593   ml_outlier:isolation_forest(score=1.00)            1.0
4858   ml_outlier:isolation_forest(score=1.00)            1.0
4859   ml_outlier:isolation_forest(score=1.00)            1.0
7311   ml_outlier:isolation_forest(score=1.00)            1.0
10738  ml_outlier:isolation_forest(score=1.00)            1.0
10950  ml_outlier:isolation_forest(score=1.00)            1.0
11058  ml_outlier:isolation_forest(score=1.00)            1.0
11062  ml_outlier:isolation_forest(score=1.00)            1.0
11066  ml_outlier:isolation_forest(score=1.00)            1.0


## Combined summary across all sources

In [13]:
summary_rows = []
for source, (history, flags) in results.items():
    summary_rows.append(
        {
            "source": source,
            "rows_scanned": len(history),
            "rows_flagged": len(flags),
            "flag_rate": len(flags) / max(len(history), 1),
            "ml_model_trained": ml_anomaly.load_local(source) is not None,
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("source").reset_index(drop=True)
summary_df

,source,rows_scanned,rows_flagged,flag_rate,ml_model_trained
0,aemo_nem,1358005,6937,0.005108,True
1,aemo_wem,211104,110,0.000521,True
2,bom,237570,204,0.000859,True
3,openelectricity,1270957,7813,0.006147,True


## Synthetic edge cases

Real history is good for "does this behave sanely at real volume," but
won't reliably contain a clean, isolated example of each signal firing
on its own. Inject both explicitly to confirm each still fires under
the *current* thresholds (`_Z_SCORE_THRESHOLD = 4.0`,
`ANOMALY_SCORE_THRESHOLD = 0.7`, `_MIN_ANOMALY_SCORE_TO_FLAG = 0.98`) —
using `aemo_nem`'s real trained model for the ML case, so this is
exercising the actual on-disk artifact, not a mock.

In [14]:
import numpy as np

# Statistical signal: stats are computed from the batch *including* the
# outlier itself (anomaly.py's own design -- a per-batch check, no
# separate baseline), so a single extreme point also inflates its own
# std and self-limits z. Needs both a large-enough sample and a big
# enough outlier to still clear the combined gate (>0.98): z_signal_score
# = min(1.0, z / (2 * 4.0)) must exceed 0.98, i.e. z > ~7.84.
rng = np.random.default_rng(0)
normal_vals = rng.normal(8000, 100, 100).tolist()
stat_df = pd.DataFrame(
    {
        "region": ["NSW1"] * 101,
        "demand_mw": normal_vals + [100000.0],  # far outside this batch's own spread
        "price_mwh": [80.0] * 101,
    }
)
stat_flags = anomaly.detect_anomalies(stat_df, "aemo_nem")
assert len(stat_flags) == 1
assert "statistical_outlier:demand_mw" in stat_flags.iloc[0]["anomaly_reason"]
print("statistical signal: OK ->", stat_flags.iloc[0]["anomaly_reason"], "score=", round(stat_flags.iloc[0]["anomaly_score"], 3))

# ML signal: real aemo_nem model (61,080 real training rows). Checked
# directly against ml_anomaly.score first: a large demand/price swing
# alone (e.g. demand 100,000 / price -5,000) only scores ~0.70 -- right
# at ANOMALY_SCORE_THRESHOLD but nowhere near the 0.98 combined gate.
# Near-zero demand paired with a strongly positive price -- a genuinely
# rare real-world combination -- is what actually isolates fully (score
# ~2.0 on the raw threshold->floor scale, clipped to 1.0) against this
# model's real training distribution.
ml_df = pd.DataFrame(
    {
        "region": ["NSW1"] * 6,
        "demand_mw": [8100.0, 7950.0, 8200.0, 8050.0, 7900.0, 0.0],
        "price_mwh": [78.0, 81.0, 79.0, 80.0, 77.0, 5000.0],
    }
)
ml_flags = anomaly.detect_anomalies(ml_df, "aemo_nem")
assert len(ml_flags) == 1
assert "ml_outlier:isolation_forest" in ml_flags.iloc[0]["anomaly_reason"]
print("ML signal: OK ->", ml_flags.iloc[0]["anomaly_reason"], "score=", round(ml_flags.iloc[0]["anomaly_score"], 3))

statistical signal: OK -> statistical_outlier:demand_mw(z=9.9) score= 1.0
ML signal: OK -> ml_outlier:isolation_forest(score=1.00) score= 1.0


## Real DB round-trip — `meta.anomalies`, with deliberate cleanup

`anomaly.record_anomalies`/`anomaly.count_anomalies` — the real
functions, not a hand-copied version — write to the real Neon
`meta.anomalies` table (`pipeline.tasks._common.standard_run` calls
these after every real ingest run). This inserts real flagged rows from
the end-to-end run above under one fixed, recognisable `run_id`,
verifies they landed, then **deletes them immediately** — proving the
write path works end to end without leaving any residue in the shared
table.

In [15]:
import uuid

from sqlalchemy import text

from app.db.session import get_session

_TEST_RUN_ID = uuid.UUID("00000000-0000-4000-8000-000000000001")  # fixed, recognisable
_TABLES = {
    "bom": "bom_observations",
    "aemo_nem": "aemo_nem_dispatch",
    "aemo_wem": "aemo_wem_dispatch",
    "openelectricity": "openelectricity_mix",
}


async def _meta_anomalies_exists() -> bool:
    async with get_session() as session:
        result = await session.execute(
            text(
                "SELECT 1 FROM information_schema.tables "
                "WHERE table_schema = 'meta' AND table_name = 'anomalies'"
            )
        )
        return result.first() is not None


async def _cleanup_test_run() -> None:
    # Child (meta.anomalies) before parent (meta._ingest_log) --
    # meta.anomalies.run_id has a real FK against meta._ingest_log.id.
    async with get_session() as _session:
        await _session.execute(
            text("DELETE FROM meta.anomalies WHERE run_id = :run_id"),
            {"run_id": str(_TEST_RUN_ID)},
        )
        await _session.execute(
            text("DELETE FROM meta._ingest_log WHERE id = :run_id"),
            {"run_id": str(_TEST_RUN_ID)},
        )


if not await _meta_anomalies_exists():
    print("SKIPPED: meta.anomalies does not exist in this database right now.")
else:
    # Prefer a real flagged batch from the end-to-end run above; fall
    # back to the synthetic ML case (still real detect_anomalies output,
    # just not sourced from real history) if every real source's
    # high-confidence gate happened to flag nothing this run.
    real_source, real_flags = next(
        ((s, f) for s, (_, f) in results.items() if not f.empty), (None, None)
    )
    if real_source is not None:
        test_source, test_table, test_flags = real_source, _TABLES[real_source], real_flags.head(3)
        print(f"using {len(test_flags)} real flagged row(s) from {real_source!r}")
    else:
        test_source, test_table, test_flags = "aemo_nem", "aemo_nem_dispatch", ml_flags
        print("no real source flagged anything this run -- using the synthetic ML case instead")

    await _cleanup_test_run()  # in case a previous interrupted run left residue

    # meta.anomalies.run_id FKs to meta._ingest_log.id -- a real ingest
    # run always logs a row there first; this test row mirrors that,
    # tagged unmistakably as a test (source='anomaly_detection_notebook_test').
    async with get_session() as _session:
        await _session.execute(
            text(
                "INSERT INTO meta._ingest_log (id, source, status, triggered_by) "
                "VALUES (:id, :source, 'success', 'anomaly_detection_notebook_test')"
            ),
            {"id": str(_TEST_RUN_ID), "source": "anomaly_detection_notebook_test"},
        )

    before = await anomaly.count_anomalies(_TEST_RUN_ID)
    assert before == 0

    await anomaly.record_anomalies(_TEST_RUN_ID, test_source, test_table, test_flags)

    after = await anomaly.count_anomalies(_TEST_RUN_ID)
    print(f"inserted: {after} row(s) under test run_id {_TEST_RUN_ID}")
    assert after == len(test_flags)

    # Cleanup -- never leave synthetic/test rows behind in the real tables.
    await _cleanup_test_run()

    remaining = await anomaly.count_anomalies(_TEST_RUN_ID)
    assert remaining == 0
    print("cleanup verified: 0 rows remain under the test run_id (both meta.anomalies and meta._ingest_log).")

using 3 real flagged row(s) from 'bom'
inserted: 3 row(s) under test run_id 00000000-0000-4000-8000-000000000001
cleanup verified: 0 rows remain under the test run_id (both meta.anomalies and meta._ingest_log).


## Summary

- Every trained ML model on disk (`data/staging/models/anomaly/*.joblib`)
  loaded and its real calibration (`rows_trained`/`decision_threshold`/
  `decision_floor`) inspected.
- The real `anomaly.detect_anomalies` ran end-to-end against each
  source's full real accumulated history in `landed.duckdb` — combined
  statistical + ML signals, the real `_MIN_ANOMALY_SCORE_TO_FLAG = 0.98`
  high-confidence gate, no hand-copied logic.
- Both signals confirmed to still fire correctly under current
  thresholds via targeted synthetic cases (the ML case exercises
  `aemo_nem`'s real trained forest, not a mock).
- The real `meta.anomalies` write path (`record_anomalies`/
  `count_anomalies`, imported directly from the app) round-trips
  correctly against the live Neon database, verified and cleaned up
  immediately — no residue left behind.

**Retraining** (not run here): `ecolens-ingestion train-anomaly-model
<source>` refits against the current full `landed.duckdb` history and
re-uploads to R2 — worth re-running after a large backfill materially
changes a source's distribution, but deliberately not exercised in this
read-only notebook.